In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sudhanvasavyasachis/product-length-prediction/val_fe.parquet
/kaggle/input/datasets/sudhanvasavyasachis/product-length-prediction/test_fe.parquet
/kaggle/input/datasets/sudhanvasavyasachis/product-length-prediction/train_fe.parquet


In [4]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 42.0 MB/s eta 0:00:00:00:0100:01


In [61]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import gc
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import gc
import lightgbm as lgb

In [6]:
train = pd.read_parquet("/kaggle/input/datasets/sudhanvasavyasachis/product-length-prediction/train_fe.parquet")
val   = pd.read_parquet("/kaggle/input/datasets/sudhanvasavyasachis/product-length-prediction/val_fe.parquet")
test  = pd.read_parquet("/kaggle/input/datasets/sudhanvasavyasachis/product-length-prediction/test_fe.parquet")

In [20]:
test=test.sample(10000)

In [21]:
print(f"Train : {train.shape}")
print(f"Val   : {val.shape}")
print(f"Test  : {test.shape}")

Train : (60000, 5)
Val   : (5000, 5)
Test  : (10000, 4)


In [22]:
val

,PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,CLEAN_TEXT,PRODUCT_LENGTH_LOG
0,1923654,3314,600.000000,antarctica tactical sling bag pack military sh...,6.398595
1,2550771,6426,124.015748,chitra electricals ce lights 7w led ceiling do...,4.828440
2,82131,59,665.353000,study and master technology senior phase study...,6.501820
3,293166,6044,600.000000,a business history of india enterprise and the...,6.398595
4,2125440,12558,629.920000,"zcdaye marble case for samsung galaxy a42 5g,s...",6.447179
...,...,...,...,...,...
4995,589378,6143,600.000000,resisting medical tyranny why the covid-19 man...,6.398595
4996,138377,22,964.565000,sleep bedtime reader bedtime reading,6.872713
4997,2271174,12739,393.700787,mg brand unicorn sipper round shape theme wate...,5.978128
4998,1416891,2300,622.046000,"renewed apple iphone 7 plus mn4p2hn a silver, ...",6.434620


In [23]:
corpus_df, query_df = train_test_split(
    train,
    test_size=0.25,       # 25% → ~15k query rows for LightGBM training
    random_state=42,
    shuffle=True
)

In [24]:
corpus_df = corpus_df.reset_index(drop=True)
query_df  = query_df.reset_index(drop=True)

In [25]:
print(f"\nCorpus (search database) : {len(corpus_df):,} rows")
print(f"Query  (LightGBM train)  : {len(query_df):,} rows")


Corpus (search database) : 45,000 rows
Query  (LightGBM train)  : 15,000 rows


In [26]:
MODELS = {
    "minilm"  : "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "e5_small": "intfloat/multilingual-e5-small",
}

K_PER_MODEL = 50   # 50 neighbours per model = 100 total features

In [27]:
def encode(model, texts, batch_size=256, prefix=""):
    if prefix:
        texts = [prefix + t for t in texts]
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

In [28]:
def build_faiss_index(embeddings, use_gpu=True):
    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    if use_gpu:
        try:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            print("  Using GPU FAISS")
        except Exception:
            print("  GPU FAISS failed, using CPU")
    index.add(embeddings)
    return index

In [19]:
def get_neighbour_lengths(indices, corpus_targets):
    """Map neighbour indices → their PRODUCT_LENGTH values"""
    return corpus_targets[indices]   # shape: (n_queries, k)

In [29]:
corpus_df

,PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,CLEAN_TEXT,PRODUCT_LENGTH_LOG
0,117102,12744,586.000000,waking partners severn house large print,6.375025
1,556690,6310,787.400000,the works of charles darwin vol 17 the various...,6.670006
2,1468932,3368,50.000000,aquamarine sterling silver lace ring size 7 fe...,3.931826
3,1978343,7271,1259.842518,asera princess stationery gift set for kids fo...,7.139535
4,2360416,1256,787.401574,"lunch box, portable environmentally friendly w...",6.670008
...,...,...,...,...,...
44995,357076,6123,550.000000,life of friedrich schiller,6.311735
44996,2275175,1,600.000000,girls just want to have fundamental rights fem...,6.398595
44997,2557609,12057,472.440944,"hhuan phone case for zte avid 579 5.45 , shock...",6.160027
44998,1700291,12064,600.000000,pikkme funky space astronaut stars black backg...,6.398595


In [30]:
# Collect feature dfs per split
query_feature_dfs = []
val_feature_dfs   = []
test_feature_dfs  = []

corpus_targets = corpus_df["PRODUCT_LENGTH"].values   # fixed reference

for model_name, model_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    model = SentenceTransformer(model_path)
    is_e5 = "e5" in model_name
    corpus_prefix = "passage: " if is_e5 else ""
    query_prefix  = "query: "   if is_e5 else ""

    # Encode corpus (45k) — the fixed database
    print("Encoding corpus (45k)...")
    corpus_emb = encode(model, corpus_df["CLEAN_TEXT"].tolist(),
                        batch_size=256, prefix=corpus_prefix)

    # Encode query split (15k) — LightGBM training rows
    print("Encoding query/train split (15k)...")
    query_emb = encode(model, query_df["CLEAN_TEXT"].tolist(),
                       batch_size=256, prefix=query_prefix)

    # Encode val
    print("Encoding val...")
    val_emb = encode(model, val["CLEAN_TEXT"].tolist(),
                     batch_size=256, prefix=query_prefix)

    # Encode test
    print("Encoding test...")
    test_emb = encode(model, test["CLEAN_TEXT"].tolist(),
                      batch_size=256, prefix=query_prefix)

    # Build index on corpus ONLY
    print("Building FAISS index on corpus...")
    index = build_faiss_index(corpus_emb)

    # Search all splits against corpus
    print("KNN search: query split vs corpus...")
    query_indices = index.search(query_emb, K_PER_MODEL)[1]

    print("KNN search: val vs corpus...")
    val_indices = index.search(val_emb, K_PER_MODEL)[1]

    print("KNN search: test vs corpus...")
    test_indices = index.search(test_emb, K_PER_MODEL)[1]

    # Map to PRODUCT_LENGTH values
    cols = [f"{model_name}_nn_{i}" for i in range(1, K_PER_MODEL + 1)]

    query_feature_dfs.append(pd.DataFrame(
        get_neighbour_lengths(query_indices, corpus_targets), columns=cols))
    val_feature_dfs.append(pd.DataFrame(
        get_neighbour_lengths(val_indices, corpus_targets), columns=cols))
    test_feature_dfs.append(pd.DataFrame(
        get_neighbour_lengths(test_indices, corpus_targets), columns=cols))

    del model, corpus_emb, query_emb, val_emb, test_emb, index
    gc.collect()


Model: minilm


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding val...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Encoding test...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Building FAISS index on corpus...
  Using GPU FAISS
KNN search: query split vs corpus...
KNN search: val vs corpus...
KNN search: test vs corpus...


In [35]:
print("\nConcatenating features...")

AUX_COLS = [
    "PRODUCT_TYPE_ID"   
]


Concatenating features...


In [36]:
# Query split → LightGBM train
query_meta = pd.concat([
    pd.concat(query_feature_dfs, axis=1).reset_index(drop=True),
    query_df[AUX_COLS].reset_index(drop=True)
], axis=1)
query_meta["PRODUCT_LENGTH"]     = query_df["PRODUCT_LENGTH"].values
query_meta["PRODUCT_LENGTH_LOG"] = query_df["PRODUCT_LENGTH_LOG"].values

In [37]:
# Val
val_meta = pd.concat([
    pd.concat(val_feature_dfs, axis=1).reset_index(drop=True),
    val[AUX_COLS].reset_index(drop=True)
], axis=1)
val_meta["PRODUCT_LENGTH"]     = val["PRODUCT_LENGTH"].values
val_meta["PRODUCT_LENGTH_LOG"] = val["PRODUCT_LENGTH_LOG"].values


In [49]:
test

,PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,CLEAN_TEXT
62253,2443824,6548,75.000000,"canvas wall art 36 w x 18 h, canvasart03 canva..."
65319,2841816,7617,940.000000,"custom-fit for infiniti seat belt covers, 2pac..."
91713,2695283,1396,472.440944,cozy home premium handmade ceramic milk and co...
18680,2617096,2981,590.551180,worldcare women fitness yoga sets lady sports ...
59901,706441,6095,586.613000,englisch in livingston ausgewaehlte sprachlich...
...,...,...,...,...
58963,2901732,12064,669.291338,natasha hard plastic for girls boys printed ba...
93571,1237035,12157,78.740157,jvc hafr325w white premium sound in ear headph...
4293,449950,1,175.000000,websters new world dictionary 2000
62084,2111880,3650,1700.000000,night cat camping hammock tent with mosquito n...


In [50]:
# Test
test_meta = pd.concat([
    pd.concat(test_feature_dfs, axis=1).reset_index(drop=True),
    test[AUX_COLS].reset_index(drop=True)
], axis=1)
test_meta["PRODUCT_LENGTH"]     = test["PRODUCT_LENGTH"].values
test_meta["PRODUCT_LENGTH_LOG"] = np.log1p(test["PRODUCT_LENGTH"].values)

In [51]:
print(f"\nFinal shapes:")
print(f"  query_meta (LightGBM train) : {query_meta.shape}")
print(f"  val_meta                    : {val_meta.shape}")
print(f"  test_meta                   : {test_meta.shape}")



Final shapes:
  query_meta (LightGBM train) : (15000, 103)
  val_meta                    : (5000, 103)
  test_meta                   : (10000, 103)


In [52]:
print("\nSanity check — correlation of mean neighbour length with target:")
nn_cols = [c for c in query_meta.columns if "_nn_" in c]
query_meta["mean_nn"] = query_meta[nn_cols].mean(axis=1)
corr = query_meta["mean_nn"].corr(query_meta["PRODUCT_LENGTH"])
print(f"  Corr(mean_nn, PRODUCT_LENGTH) = {corr:.4f}")
query_meta.drop(columns=["mean_nn"], inplace=True)


Sanity check — correlation of mean neighbour length with target:
  Corr(mean_nn, PRODUCT_LENGTH) = 0.5670


In [53]:
from sklearn.model_selection import KFold
from sklearn import metrics
import warnings
warnings.filterwarnings("ignore")

In [72]:
# Combine query + val for training
train_lgbm = pd.concat([query_meta, val_meta], axis=0).reset_index(drop=True)
print(f"Train (query + val) : {train_lgbm.shape}")
print(f"Test (held out)     : {test_meta.shape}")

Train (query + val) : (20000, 103)
Test (held out)     : (10000, 104)


In [73]:
CLIP_MIN = 10   # anything below 10 gets treated as 10
train_lgbm["PRODUCT_LENGTH_CLIPPED"] = train_lgbm["PRODUCT_LENGTH"].clip(lower=CLIP_MIN)
test_meta["PRODUCT_LENGTH_CLIPPED"]  = test_meta["PRODUCT_LENGTH"].clip(lower=CLIP_MIN)

In [74]:
DROP_COLS = ["PRODUCT_LENGTH", "PRODUCT_LENGTH_LOG"]
FEATURE_COLS = [c for c in train_lgbm.columns if c not in DROP_COLS]
CAT_COLS = ["PRODUCT_TYPE_ID"]

X      = train_lgbm[FEATURE_COLS]
y_log  = train_lgbm["PRODUCT_LENGTH_LOG"]   # train on log    # for metric reporting
y      = train_lgbm["PRODUCT_LENGTH_CLIPPED"]
y_real = train_lgbm["PRODUCT_LENGTH"]          # original for final metric reporting
X_test      = test_meta[FEATURE_COLS]
y_test_real = test_meta["PRODUCT_LENGTH"]

print(f"\nFeatures : {len(FEATURE_COLS)}")
print(f"Target   : PRODUCT_LENGTH_LOG")



Features : 102
Target   : PRODUCT_LENGTH_LOG


In [75]:
def mape_score(actual, predicted):
    """Standard MAPE"""
    return metrics.mean_absolute_percentage_error(actual, predicted)

def competition_score(actual, predicted):
    """max(0, 100*(1 - MAPE))"""
    mape = mape_score(actual, predicted)
    return max(0, 100 * (1 - mape))

In [76]:
params = {
    "objective"        : "mape",
    "metric"           : "mape",
    "n_estimators"     : 5000,
    "learning_rate"    : 0.05,
    "num_leaves"       : 127,
    "max_depth"        : -1,
    "min_child_samples": 20,
    "subsample"        : 0.8,
    "subsample_freq"   : 1,
    "colsample_bytree" : 0.8,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 0.1,
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1,
}

In [77]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_log  = np.zeros(len(train_lgbm))   # out of fold predictions (log scale)
test_preds_log = np.zeros(len(test_meta))    # test predictions averaged across folds

In [78]:
fold_mapes  = []
fold_scores = []

print("\n" + "="*60)
print("5-Fold Cross Validation")
print("="*60)


5-Fold Cross Validation


In [79]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold+1}/5 ...")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

    # actual original scale values for metric
    y_val_real = y_real.iloc[val_idx].values

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y.iloc[train_idx],
        eval_set=[(X_val, y.iloc[val_idx])],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=500)
        ],
        categorical_feature=CAT_COLS
    )
    
    val_preds  = model.predict(X_val)
    val_preds  = np.maximum(val_preds, 0)   # clip negatives
    
    # report on original unclipped values
    y_val_real = y_real.iloc[val_idx].values
    fold_mape  = mape_score(y_val_real, val_preds)
    fold_score = competition_score(y_val_real, val_preds)


    fold_mapes.append(fold_mape)
    fold_scores.append(fold_score)

    print(f"  Best iteration : {model.best_iteration_}")
    print(f"  MAPE           : {fold_mape:.4f}  ({fold_mape*100:.2f}%)")
    print(f"  Competition    : {fold_score:.4f}")


Fold 1/5 ...
[500]	valid_0's mape: 0.11204
[1000]	valid_0's mape: 0.111182
[1500]	valid_0's mape: 0.110849
  Best iteration : 1657
  MAPE           : 0.2401  (24.01%)
  Competition    : 75.9891

Fold 2/5 ...
[500]	valid_0's mape: 0.0874495
[1000]	valid_0's mape: 0.0866435
[1500]	valid_0's mape: 0.0863234
[2000]	valid_0's mape: 0.0861467
  Best iteration : 1956
  MAPE           : 0.1812  (18.12%)
  Competition    : 81.8793

Fold 3/5 ...
[500]	valid_0's mape: 0.085483
[1000]	valid_0's mape: 0.0846965
[1500]	valid_0's mape: 0.0843514
[2000]	valid_0's mape: 0.0841621
[2500]	valid_0's mape: 0.0840093
  Best iteration : 2792
  MAPE           : 0.2135  (21.35%)
  Competition    : 78.6521

Fold 4/5 ...
[500]	valid_0's mape: 0.0902088
[1000]	valid_0's mape: 0.0894597
[1500]	valid_0's mape: 0.0891258
[2000]	valid_0's mape: 0.0888589
[2500]	valid_0's mape: 0.088709
[3000]	valid_0's mape: 0.0885377
  Best iteration : 2977
  MAPE           : 0.2171  (21.71%)
  Competition    : 78.2881

Fold 5/5 ..

In [86]:
# ══════════════════════════════════════════════
# FINAL MODEL — TRAIN ON FULL 20K
# ══════════════════════════════════════════════
print("\n" + "="*60)
print("FINAL MODEL — TRAINING ON FULL 20K")
print("="*60)

final_n_estimators = 1000

final_params = params.copy()
final_params["n_estimators"] = final_n_estimators

final_model = lgb.LGBMRegressor(**final_params)
final_model.fit(
    X, y,
    categorical_feature=CAT_COLS,
    callbacks=[lgb.log_evaluation(period=200)]
)



FINAL MODEL — TRAINING ON FULL 20K


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, metric='mape',
              n_estimators=1000, n_jobs=-1, num_leaves=127, objective='mape',
              random_state=42, reg_alpha=0.1, reg_lambda=0.1, subsample=0.8,
              subsample_freq=1, verbose=-1)

In [87]:

# Predict on test
final_test_preds = np.maximum(final_model.predict(X_test), 0)

test_mape_final  = mape_score(y_test_real.values, final_test_preds)
test_score_final = competition_score(y_test_real.values, final_test_preds)

print(f"\nTest MAPE           : {test_mape_final*100:.2f}%")
print(f"Test Competition    : {test_score_final:.4f}")


Test MAPE           : 11.57%
Test Competition    : 88.4258
